In [12]:
import pandas as pd
import numpy as np
import glob
import os

In [13]:
DATA_PATH = '/content/drive/MyDrive/all_data_nir/*.csv'


In [14]:
# AOI progress bar
PROGRESS_BAR_AOI_SMALL = {
    'x1': 760,
    'x2': 1800,
    'y1': 600,
    'y2': 680
}
PROGRESS_BAR_AOI_LARGE = {
    'x1': 760,
    'x2': 1800,
    'y1': 340,
    'y2': 460
}


In [15]:
# радиус фиксации
FIXATION_RADIUS = 35
def inside_aoi(x, y, aoi):
    """
    Проверка попадания взгляда в AOI
    """

    return (
        aoi['x1'] <= x <= aoi['x2'] and
        aoi['y1'] <= y <= aoi['y2']
    )


def calculate_scanpath(df):
    """
    Scanpath Length
    """

    distances = []

    for i in range(1, len(df)):

        dx = df.iloc[i]['x'] - df.iloc[i - 1]['x']
        dy = df.iloc[i]['y'] - df.iloc[i - 1]['y']

        distance = np.sqrt(dx**2 + dy**2)

        distances.append(distance)

    return np.sum(distances)


def calculate_saccade_velocity(df):
    """
    Скорость движения взгляда
    """

    velocities = []

    for i in range(1, len(df)):

        dx = df.iloc[i]['x'] - df.iloc[i - 1]['x']
        dy = df.iloc[i]['y'] - df.iloc[i - 1]['y']

        distance = np.sqrt(dx**2 + dy**2)

        dt = (
            df.iloc[i]['time_from_form_ms'] -
            df.iloc[i - 1]['time_from_form_ms']
        )

        if dt > 0:
            velocity = distance / dt
            velocities.append(velocity)

    if len(velocities) == 0:
        return 0

    return np.mean(velocities)


def calculate_fixations(df):
    """
    Поиск фиксаций
    """

    fixation_count = 0
    fixation_durations = []

    current_fixation_time = 0

    for i in range(1, len(df)):

        dx = df.iloc[i]['x'] - df.iloc[i - 1]['x']
        dy = df.iloc[i]['y'] - df.iloc[i - 1]['y']

        distance = np.sqrt(dx**2 + dy**2)

        dt = (
            df.iloc[i]['time_from_form_ms'] -
            df.iloc[i - 1]['time_from_form_ms']
        )

        # если взгляд почти не двигается
        if distance < FIXATION_RADIUS:

            current_fixation_time += dt

        else:

            # завершение фиксации
            if current_fixation_time > 0:

                fixation_count += 1
                fixation_durations.append(current_fixation_time)

            current_fixation_time = 0

    avg_fixation_duration = 0

    if len(fixation_durations) > 0:
        avg_fixation_duration = np.mean(fixation_durations)

    return fixation_count, avg_fixation_duration


def calculate_progress_metrics(df):
    """
    Метрики progress bar
    """

    # точки внутри progress bar
    progress_df = df[df['on_progress_bar']]

    # =====================================================
    # Number of Fixations на progress bar
    # =====================================================

    progress_fixations = len(progress_df)

    # =====================================================
    # Fixation Duration
    # =====================================================

    fixation_time = progress_df['dt'].sum()

    # =====================================================
    # Attention Ratio
    # =====================================================

    total_time = df['dt'].sum()

    attention_ratio = 0

    if total_time > 0:
        attention_ratio = fixation_time / total_time

    return (
        progress_fixations,
        fixation_time,
        attention_ratio
    )


def calculate_revisits(df):
    """
    Revisits к полям формы
    """

    revisits = 0

    previous_state = False

    for current_state in df['on_progress_bar']:

        # пользователь вернулся к progress bar
        if current_state and not previous_state:
            revisits += 1

        previous_state = current_state

    return revisits


In [16]:
files = glob.glob(DATA_PATH)

all_metrics = []

for file in files:

    print(f'Обработка: {file}')

    # =====================================================
    # ЗАГРУЗКА
    # =====================================================

    df = pd.read_csv(file)

    # сортировка по времени
    df = df.sort_values('time_from_form_ms')

    # =====================================================
    # ВРЕМЯ МЕЖДУ ТОЧКАМИ
    # =====================================================

    df['dt'] = (
        df['time_from_form_ms']
        .diff()
        .fillna(0)
    )

    # =====================================================
    # ИНФОРМАЦИЯ О ПОЛЬЗОВАТЕЛЕ
    # =====================================================

    participant_id = df['participantId'].iloc[0]

    form_name = df['formName'].iloc[0]

    # =====================================================
    # ПРОГРЕСС БАР
    # =====================================================

    if '_without' in form_name:
      progress_bar = 'without'

    elif '_with' in form_name:
      progress_bar = 'with'

    else:
      progress_bar = 'unknown'

    # =====================================================
    # РАЗМЕР ФОРМЫ
    # =====================================================

    if 'small' in form_name:

        form_size = 'small'

        current_aoi = (
            PROGRESS_BAR_AOI_SMALL
        )

    elif 'large' in form_name:

        form_size = 'large'

        current_aoi = (
            PROGRESS_BAR_AOI_LARGE
        )

    else:

        form_size = 'unknown'

        current_aoi = (
            PROGRESS_BAR_AOI_SMALL
        )
    # =====================================================
    # AOI progress bar
    # =====================================================

    df['on_progress_bar'] = df.apply(

        lambda row: inside_aoi(

            row['x'],
            row['y'],
            current_aoi

        ),

        axis=1
    )


    # =====================================================
    # Completion Time
    # =====================================================

    completion_time = (
        df['time_from_form_ms'].max() -
        df['time_from_form_ms'].min()
    )

    # =====================================================
    # Scanpath Length
    # =====================================================

    scanpath_length = calculate_scanpath(df)

    # =====================================================
    # Saccade Velocity
    # =====================================================

    saccade_velocity = calculate_saccade_velocity(df)

    # =====================================================
    # Общие фиксации
    # =====================================================

    fixation_count, fixation_duration = (
        calculate_fixations(df)
    )

    # =====================================================
    # Метрики progress bar
    # =====================================================

    (
        progress_fixations,
        progress_fixation_time,
        attention_ratio
    ) = calculate_progress_metrics(df)

    # =====================================================
    # Revisits
    # =====================================================

    revisits = calculate_revisits(df)

    # =====================================================
    # СОХРАНЕНИЕ МЕТРИК
    # =====================================================

    metrics = {

        'participant_id': participant_id,

        'form_name': form_name,

        'form_size': form_size,

        'progress_bar': progress_bar,

        # ==============================================
        # ОСНОВНЫЕ МЕТРИКИ
        # ==============================================

        'completion_time_ms': completion_time,

        'scanpath_length': scanpath_length,

        'saccade_velocity': saccade_velocity,

        # ==============================================
        # FIXATIONS
        # ==============================================

        'fixation_count': fixation_count,

        'avg_fixation_duration_ms': fixation_duration,

        # ==============================================
        # PROGRESS BAR
        # ==============================================

        'progress_fixations': progress_fixations,

        'progress_fixation_time_ms': progress_fixation_time,

        'attention_ratio': attention_ratio,

        'revisits': revisits
    }

    all_metrics.append(metrics)

# =========================================================
# ИТОГОВАЯ ТАБЛИЦА
# =========================================================

results_df = pd.DataFrame(all_metrics)

print(results_df.head())

# =========================================================
# СОХРАНЕНИЕ
# =========================================================

results_df.to_csv(
    'eye_tracking_metrics.csv',
    index=False
)

print('\nАнализ завершён')
print('Файл сохранён: eye_tracking_metrics.csv')

Обработка: /content/drive/MyDrive/all_data_nir/gaze_form2_44_1776668194922.csv
Обработка: /content/drive/MyDrive/all_data_nir/gaze_form2_47_1777270117549.csv
Обработка: /content/drive/MyDrive/all_data_nir/gaze_form2_49_1777270774477.csv
Обработка: /content/drive/MyDrive/all_data_nir/gaze_form2_7_1776671169003.csv
Обработка: /content/drive/MyDrive/all_data_nir/gaze_form2_74_1777271156615.csv
Обработка: /content/drive/MyDrive/all_data_nir/gaze_form2_79_1777276875941.csv
Обработка: /content/drive/MyDrive/all_data_nir/gaze_form2_52_1776667444455.csv
Обработка: /content/drive/MyDrive/all_data_nir/gaze_form2_48_1776668897849.csv
Обработка: /content/drive/MyDrive/all_data_nir/gaze_form2_45_1776671351221.csv
Обработка: /content/drive/MyDrive/all_data_nir/gaze_form2_57_1776672043328.csv
Обработка: /content/drive/MyDrive/all_data_nir/gaze_form2_48_1776668770224.csv
Обработка: /content/drive/MyDrive/all_data_nir/gaze_form2_68_1777273589186.csv
Обработка: /content/drive/MyDrive/all_data_nir/gaze_f